# Middleware

Middleware provides a way to more tightly control what happens inside the agent. 

In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018566942AE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018566AA3A40>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

## Summarization Middleware

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


# Messagebased Summarization Middleware
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
        ]
        )



In [9]:
# Run with thread id
config={"configurable": {"thread_id": "test-1"}}

In [11]:
# Alternative test data
questions = [
    "What is the capital of France?",
    "What is the largest planet in our solar system?",
    "Who wrote 'To Kill a Mockingbird'?",
    "What is the boiling point of water?",
    "What is the currency of Japan?",
    "Who painted the Mona Lisa?",
    "What is the tallest mountain in the world?",
    "What is the chemical symbol for gold?",
    "Who is the current President of the United States?",
    "What is the speed of light?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f"Q: {q}\nA: {response}\nLength: {len(response['messages'])}\n")

Q: What is the capital of France?
A: {'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='2a595442-2d0e-4a79-9ddc-06390d90ac8e'), AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'The user asks a simple factual question. Provide answer: Paris.'}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 78, 'total_tokens': 110, 'completion_time': 0.066559831, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.015721915, 'prompt_tokens_details': None, 'queue_time': 0.31422467, 'total_time': 0.082281746}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_96d96a151c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a13a22-0b2f-7aa3-9d47-5603417d68ad-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 32, 'total_tokens': 

# Human in the Loop Middleware

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """A tool that reads an email given its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(
    recipient: str,
    subject: str,
    body: str
) -> str:
    """A tool that sends an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'"

In [17]:
agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool": {
                "allowed_decisions": ["approve", "reject", "edit"]
            },
            "read_email_tool": False,
        }
    )]
)

In [19]:
config = {"configurable": {"thread_id": "test-approve"}}
#step 1
result = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to john@example.com with subject 'Hello' and body 'How are you?'")]}
    , config=config
    )

In [20]:
result

{'messages': [HumanMessage(content="Send an email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='8749bf2f-75ea-4c91-9b30-45627eb9cc21'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide body, recipient, subject.', 'tool_calls': [{'id': 'fc_5d33c711-ee2c-4f71-94b1-925cea235812', 'function': {'arguments': '{"body":"How are you?","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 175, 'total_tokens': 239, 'completion_time': 0.134046604, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.041234433, 'prompt_tokens_details': None, 'queue_time': 0.158543357, 'total_time': 0.175281037}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'finish_reason': 'tool

In [25]:
# step 2: Approve the action
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decision": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print("Resumed! Result:", result)

Paused! Approving...


KeyError: 'decisions'

In [26]:
result

{'messages': [HumanMessage(content="Send an email to john@example.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='8749bf2f-75ea-4c91-9b30-45627eb9cc21'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide body, recipient, subject.', 'tool_calls': [{'id': 'fc_5d33c711-ee2c-4f71-94b1-925cea235812', 'function': {'arguments': '{"body":"How are you?","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 175, 'total_tokens': 239, 'completion_time': 0.134046604, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.041234433, 'prompt_tokens_details': None, 'queue_time': 0.158543357, 'total_time': 0.175281037}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'finish_reason': 'tool